In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report, f1_score

In [2]:
df = pd.read_csv("../data/risk_dataset_v2.csv")

df.head()

,license,license_family,project_type,distribution_model,usage,risk,reason
0,BSD-3-Clause,Permissive,commercial closed-source,distributed,library,Low,BSD license is permissive and mainly requires ...
1,BSD-2-Clause,Permissive,commercial SaaS,hosted,library,Low,BSD-2-Clause has minimal compliance obligations
2,MIT,Permissive,internal enterprise tool,internal-only,library,Low,Internal use with MIT license is generally low...
3,Apache-2.0,Permissive,open-source project,distributed,framework,Low,Apache-2.0 is permissive and suitable for open...
4,Apache-2.0,Permissive,mobile application,distributed,library,Low,Apache license allows commercial use with attr...


In [3]:
print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nRisk distribution:")
print(df["risk"].value_counts())

Dataset shape: (61, 7)

Columns:
Index(['license', 'license_family', 'project_type', 'distribution_model',
       'usage', 'risk', 'reason'],
      dtype='str')

Risk distribution:
risk
Low       25
High      20
Medium    16
Name: count, dtype: int64


In [4]:
df["input_text"] = (
    "License: " + df["license"].astype(str) + " " +
    "License Family: " + df["license_family"].astype(str) + " " +
    "Project Type: " + df["project_type"].astype(str) + " " +
    "Distribution Model: " + df["distribution_model"].astype(str) + " " +
    "Usage: " + df["usage"].astype(str)
)

X = df["input_text"]
y = df["risk"]

df[["input_text", "risk"]].head()

,input_text,risk
0,License: BSD-3-Clause License Family: Permissi...,Low
1,License: BSD-2-Clause License Family: Permissi...,Low
2,License: MIT License Family: Permissive Projec...,Low
3,License: Apache-2.0 License Family: Permissive...,Low
4,License: Apache-2.0 License Family: Permissive...,Low


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 48
Testing samples: 13


In [6]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    max_features=1000
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Vectorized training shape:", X_train_vec.shape)
print("Vectorized testing shape:", X_test_vec.shape)

Vectorized training shape: (48, 352)
Vectorized testing shape: (13, 352)


In [7]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ),
    "Linear SVM": LinearSVC(
        class_weight="balanced"
    ),
    "Naive Bayes": MultinomialNB(),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced"
    )
}

In [8]:
results = []

for model_name, model in models.items():
    model.fit(X_train_vec, y_train)
    y_pred = model.predict(X_test_vec)

    accuracy = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro")

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Macro F1": macro_f1
    })

    print("=" * 60)
    print(model_name)
    print("=" * 60)
    print("Accuracy:", accuracy)
    print("Macro F1:", macro_f1)
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

Logistic Regression
Accuracy: 0.7692307692307693
Macro F1: 0.7666666666666666

Classification Report:
              precision    recall  f1-score   support

        High       0.67      1.00      0.80         4
         Low       1.00      0.60      0.75         5
      Medium       0.75      0.75      0.75         4

    accuracy                           0.77        13
   macro avg       0.81      0.78      0.77        13
weighted avg       0.82      0.77      0.77        13

Linear SVM
Accuracy: 0.8461538461538461
Macro F1: 0.8486772486772486

Classification Report:
              precision    recall  f1-score   support

        High       0.67      1.00      0.80         4
         Low       1.00      0.80      0.89         5
      Medium       1.00      0.75      0.86         4

    accuracy                           0.85        13
   macro avg       0.89      0.85      0.85        13
weighted avg       0.90      0.85      0.85        13

Naive Bayes
Accuracy: 0.7692307692307693
Ma

In [9]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="Macro F1", ascending=False)

results_df

,Model,Accuracy,Macro F1
1,Linear SVM,0.846154,0.848677
3,Random Forest,0.846154,0.848677
0,Logistic Regression,0.769231,0.766667
2,Naive Bayes,0.769231,0.760943


In [10]:
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

print("Best Model:", best_model_name)

Best Model: Linear SVM


In [21]:
sample = [
    "License: Proprietary License Family: weak copyleft Project Type: commercial desktop software Distribution Model: distributed Usage:Static library "
]

sample_vec = vectorizer.transform(sample)
prediction = best_model.predict(sample_vec)

print("Predicted Risk:", prediction[0])

Predicted Risk: Medium


In [18]:
joblib.dump(best_model, "../models/baseline2_best_model.pkl")
joblib.dump(vectorizer, "../models/baseline2_tfidf_vectorizer.pkl")

print("Baseline 2 best model and vectorizer saved successfully.")

Baseline 2 best model and vectorizer saved successfully.


In [19]:
results_df.to_csv("../outputs/baseline2_model_comparison_results.csv", index=False)

print("Results saved to outputs folder.")

Results saved to outputs folder.
